In [ ]:
from contextlib import nullcontext

import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download
from transformers import Sam2Model, Sam2Processor

MODEL_ID = "facebook/sam2.1-hiera-large"
# Same sample image as sam2/notebooks/images/truck.jpg in the official repository
IMAGE_PATH = hf_hub_download(repo_id="hf-internal-testing/sam2-fixtures", filename="truck.jpg", repo_type="dataset")
device = "cuda" if torch.cuda.is_available() else "cpu"
autocast_context = (
    torch.autocast("cuda", dtype=torch.bfloat16)
    if device == "cuda"
    else nullcontext()
)

# Load the SAM2 model and processor
model = Sam2Model.from_pretrained(MODEL_ID).to(device).eval()
processor = Sam2Processor.from_pretrained(MODEL_ID)
print("Model loaded")
# Load the input image
image = Image.open(IMAGE_PATH).convert("RGB")

# Compute the image embeddings once and reuse them for each prompt (equivalent to SAM2ImagePredictor.set_image)
with torch.inference_mode(), autocast_context:
    pixel_values = processor(images=image, return_tensors="pt")["pixel_values"].to(device)
    image_embeddings = model.get_image_embeddings(pixel_values)


def predict(input_points=None, input_labels=None, input_boxes=None, multimask_output=True):
    """Equivalent to SAM2ImagePredictor.predict

    Args:
        input_points: Points with shape (num_points, 2) for a single object.
        input_labels: Labels with shape (num_points,). 1 is positive and 0 is negative.
        input_boxes: Boxes with shape (4,) for a single object or (num_boxes, 4) for batch inference.

    Returns:
        masks: Binary masks with shape (3, H, W), or (num_boxes, 3, H, W) for batch inference.
        scores: Predicted IoU scores with shape (3,), or (num_boxes, 3).
        logits: Low-resolution mask logits with shape (3, 256, 256), or (num_boxes, 3, 256, 256).
    """
    input_boxes = None if input_boxes is None else np.asarray(input_boxes)
    is_batch = input_boxes is not None and input_boxes.ndim == 2
    # Convert the prompts into the nested list format of the processor: [image, object, point, xy]
    inputs = processor(
        images=image,
        input_points=None if input_points is None else [[np.asarray(input_points).tolist()]],
        input_labels=None if input_labels is None else [[np.asarray(input_labels).tolist()]],
        input_boxes=None if input_boxes is None else [input_boxes.reshape(-1, 4).tolist()],
        return_tensors="pt",
    ).to(device)
    with torch.inference_mode(), autocast_context:
        outputs = model(
            image_embeddings=image_embeddings,
            input_points=inputs.get("input_points"),
            input_labels=inputs.get("input_labels"),
            input_boxes=inputs.get("input_boxes"),
            multimask_output=multimask_output,
        )
    # Upscale the masks to the original image size and binarize them
    masks = processor.post_process_masks(outputs.pred_masks.float().cpu(), inputs["original_sizes"].cpu())[0].numpy()
    scores = outputs.iou_scores[0].float().cpu().numpy()
    logits = outputs.pred_masks[0].float().cpu().numpy()
    if not is_batch:
        masks, scores, logits = masks[0], scores[0], logits[0]
    return masks, scores, logits


# Display the sorted masks, scores, and input points/boxes
def show_masks(image, masks, scores,
               input_points=None, input_labels=None, input_box=None,
               max_shown=3, fig_title=None):
    if (input_points is None) ^ (input_labels is None):
        raise ValueError("Both input_labels and input_points should be provided")
    if input_points is None:
        input_points = []
    if input_labels is None:
        input_labels = []
    if input_box is None:
        input_box = []
    # Sort masks by scores in descending order
    sorted_ind = np.argsort(scores)[::-1]
    masks = masks[sorted_ind]
    scores = scores[sorted_ind]
    num_masks_to_show = min(max_shown, len(sorted_ind))

    # Display each mask with the corresponding score and input points/boxes
    fig, axes = plt.subplots(1, num_masks_to_show, figsize=(num_masks_to_show * 5, 5))
    if num_masks_to_show == 1:
        axes = [axes]
    for i in range(num_masks_to_show):
        mask_overlay = np.zeros((*masks[i].shape, 4), dtype=np.float32)
        mask_overlay[masks[i] > 0] = [30 / 255, 144 / 255, 1.0, 0.5]
        axes[i].imshow(image)
        axes[i].imshow(mask_overlay)
        for input_point, input_label in zip(input_points, input_labels):
            axes[i].scatter(
                input_point[0],
                input_point[1],
                color="yellow" if input_label == 1 else "red",
                marker="*" if input_label == 1 else "o",
                s=250,
                edgecolor="black",
            )
        if len(input_box) == 4:
            rect = plt.Rectangle(
                (input_box[0], input_box[1]),
                input_box[2] - input_box[0],
                input_box[3] - input_box[1],
                linewidth=2,
                edgecolor="green",
                facecolor="none",
            )
            axes[i].add_patch(rect)
        axes[i].axis("off")
        axes[i].set_title(f"Mask {i+1}, Score: {scores[i]:.4f}")

    if fig_title is not None:
        fig.suptitle(fig_title)
        plt.tight_layout()
    plt.show()

In [ ]:
### Inference with one positive point input
# Prepare the input point and label
input_points = np.array([[image.width / 2, image.height / 2]])
input_labels = np.array([1])
# Inference
masks, scores, logits = predict(
    input_points=input_points,
    input_labels=input_labels,
    multimask_output=True,
)
# Show the result
print(f"masks: {masks}, scores: {scores}, logits: {logits}")
print(f"masks_shape: {masks.shape}, scores_shape: {scores.shape}, logits_shape: {logits.shape}")

show_masks(image, masks, scores, input_points=input_points, input_labels=input_labels)

In [ ]:
### Inference with positive and negative point input
# Prepare the input points and labels
input_points = np.array([[image.width / 2, image.height / 2],
                         [image.width / 4, image.height / 2]])
input_labels = np.array([1, 0])
# Inference
masks, scores, logits = predict(
    input_points=input_points,
    input_labels=input_labels,
    multimask_output=True,
)
# Show the result
print(f"masks: {masks}, scores: {scores}, logits: {logits}")
print(f"masks_shape: {masks.shape}, scores_shape: {scores.shape}, logits_shape: {logits.shape}")

show_masks(image, masks, scores, input_points=input_points, input_labels=input_labels)

In [ ]:
### Inference with bounding box
# Prepare the input bounding box
input_box = np.array([425, 600, 700, 875])
# Inference
masks, scores, logits = predict(
    input_boxes=input_box,
    multimask_output=True,
)
# Show the result
print(f"masks: {masks}, scores: {scores}, logits: {logits}")
print(f"masks_shape: {masks.shape}, scores_shape: {scores.shape}, logits_shape: {logits.shape}")

show_masks(image, masks, scores, input_box=input_box)

In [ ]:
### Batch inference with bounding boxes
# Prepare the input bounding boxes
input_boxes = np.array([
    [75, 275, 1725, 850],
    [425, 600, 700, 875],
    [1375, 550, 1650, 800],
    [1240, 675, 1400, 750],
])
masks_batch, scores_batch, _ = predict(
    input_boxes=input_boxes,
    multimask_output=True,
)
print(f"masks_shape: {masks_batch.shape}, scores_shape: {scores_batch.shape}")
for masks, scores, input_box in zip(masks_batch, scores_batch, input_boxes):
    sorted_ind = np.argsort(scores)[::-1]
    scores = scores[sorted_ind]
    masks = masks[sorted_ind]
    show_masks(image, masks, scores, input_box=input_box, fig_title=f"Input box: {input_box}")